# Contextual Embeddings and the Masked-LM Objective

Word2vec gives every word type one vector, so a word with two senses gets one point for both, and a word whose news
depends on its neighbor gets one point for every neighbor. A **contextual** embedding gives each occurrence its own
vector, a function of the whole sentence. BERT trains such vectors by hiding some tokens and asking the encoder to put
them back: the **masked-language-model objective**.

We ask four questions of that objective. What does it converge to? (The posterior of the hidden token given the
corrupted sentence, in closed form.) Is the result a language model? (No: its conditionals belong to no joint
distribution.) What do the vectors look like? (Occurrences of one word spread out with depth, inside a cone.) And what
can they do that a static vector cannot? (Tell a company from its product, and good news from bad, by reading the
neighbors.)

This notebook imports the topic's module, which owns every number; each cell calls a function and prints what the page
quotes.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "contextual_embeddings_masked_lm.py").exists()
                       else here / "notebooks" / "contextual-embeddings-masked-lm"))
import contextual_embeddings_masked_lm as cm

S = cm.summary()
print("kestrel, ORG or PRODUCT (5-fold CV):  static layer 0", round(S["alias_kestrel_cv_layer0"], 3),
      "| layer 1", round(S["alias_kestrel_cv_layer1"], 3), "| majority", round(S["alias_kestrel_majority"], 3))
print("the news a direction word reports:     contextual", round(S["news_layer1"], 3),
      "| static cap", round(S["news_cap"], 3))

## 1. One vector per type

**Def 1.** A *static* embedding is a map $x_i \mapsto e(x_i)$ that depends on the token alone. A *contextual* embedding
is a map $(x_1, \dots, x_n, i) \mapsto h_i(x_1, \dots, x_n)$. An encoder's layer 0, $E[x_i] + P_i$, is static at a fixed
position.

Our controlled language is an HMM with two regimes of four hidden states. One token, $w$, is emitted by the heaviest state
of each regime, so it has two senses that occur almost equally often. Because the HMM is a known distribution,
forward–backward gives every conditional exactly.

In [ ]:
T, B, pi = cm.hmm()
print("sense states", cm.sense_states(), "| stationary masses", np.round(pi[list(cm.sense_states())], 3))
q = cm.neighbor_pmi()
print("sense shares pi_k:", np.round(q["share"], 3))
print("PMI(w, next token) for three next tokens:", np.round(q["type"][[1, 4, 22]], 3))
print("  the two senses' PMI with them:", np.round(q["senses"][:, [1, 4, 22]], 3))

**Prop 1 (a static vector sees the mixture).** If $w$ has senses $s_k$ with shares $\pi_k$, then
$\mathrm{PMI}(w, c) = \log \sum_k \pi_k \exp \mathrm{PMI}(s_k, c)$. Hence
$\max_k (\log \pi_k + \mathrm{PMI}(s_k, c)) \le \mathrm{PMI}(w, c) \le \max_k \mathrm{PMI}(s_k, c)$. SGNS factorizes shifted
PMI (Topic 7, Thm 1), so its vector for $w$ fits this log-mixture: near the sense that dominates each context, and
never both.

**Cor 1.** Any static map gives every occurrence of $w$ the same vector, so a sense probe on it makes one prediction for
all of them. Its accuracy is therefore at most $\max_k \pi_k$.

In [ ]:
cm.test_mixture_identity()
cm.test_static_cap()
r = cm.hmm_probe()
print("static cap max_k pi_k =", round(r["static_cap"], 6), "| the token table's probe:", round(r["acc_static"], 6))

## 2. The masked-LM objective

**Def 2 (the corruption channel).** Each position is *selected* independently with probability $\rho$. A selected
position becomes `[MASK]` with probability $m$, a uniformly random token with probability $r$, and stays itself with
probability $\kappa = 1 - m - r$. BERT's values are $\rho = 0.15$, $(m, r, \kappa) = (0.8, 0.1, 0.1)$. The loss is
$-\log q(x_i \mid \tilde{x})$ averaged over the selected positions.

**Thm 1 (MLM is a proper scoring rule).** The population risk at a selected position equals
$\mathbb{E}\,H(P(\cdot \mid \tilde x, i \in M)) + \mathbb{E}\,\mathrm{KL}(P(\cdot \mid \tilde x, i \in M)\,\|\,q)$. It is
minimized exactly when $q$ is the posterior of the original token given the corrupted sequence. Gibbs' inequality makes
the KL term nonnegative, and zero only at equality.

In [ ]:
cm.test_proper_scoring()
f = cm.hmm_fit()
print(f"held-out cross-entropy at selected positions: exact Bayes {f['ce_exact']:.6f}, encoder {f['ce_model']:.6f}")
print(f"mean KL(exact || encoder) {f['kl']:.6f}  (entropy {f['entropy']:.6f})")
print({k: round(f[k], 6) for k in ("kl_mask", "kl_random", "kl_kept")}, {k: f[k] for k in ("n_mask", "n_random", "n_kept")})

**Thm 2 (the Bayes-optimal predictor under 80/10/10).** Let $p = P(x_i \mid \tilde x_{-i})$ be what the context alone
predicts. Then:
- at a `[MASK]`, $q = p$;
- at a visible token $v$, $q(y) = p(y)\,(\kappa\,\mathbb{1}[y = v] + r/V)\,/\,(\kappa\,p(v) + r/V)$.

The selection rate $\rho$ does not appear. If $r = 0$, then $q(v) = 1$: the model learns to copy the visible token. If
$\kappa = 0$, then $q = p$: the visible token tells the model nothing.

The check below compares the formula with brute-force enumeration of a length-3 joint, with the selection made
explicit. Then we read the shrinkage off the trained encoder: at a kept token, its probability of that token against
the formula applied to its own `[MASK]` prediction.

In [ ]:
cm.test_bayes_80_10_10()
print("Thm 2 = brute force to 1e-12 at rho = 0.15 and 0.4; r = 0 copies, kappa = 0 ignores")
print(f"kept tokens: encoder's q(v) {f['shrink_model_mean']:.6f}, Thm 2 from its own [MASK] guess "
      f"{f['shrink_pred_mean']:.6f} (mean abs error {f['shrink_mae']:.6f}); the [MASK] guess alone {f['shrink_masked_mean']:.6f}")

**The encoder.** Topic 11's pre-LN block without the causal mask, with three additions from BERT: a learned
absolute position table (Topic 10, Def 1), tied input and output embeddings, and a final LayerNorm. It has two blocks.
The HMM encoder has $d = 32$ and trains for 3000 Adam steps on fresh corrupted sequences of length 24. The gradient is
written by hand, checked against central differences, and the training loss forms logits only at the selected
rows.

In [ ]:
cm.test_fast_mlp_backward()
cm.test_mlm_gradients()
cm.test_hmm_mlm_matches_bayes()
m = cm.train_mlm(cm.HMM_MLM)
print("training loss every 100 steps (first, last):", round(m.loss[0], 3), round(m.loss[-1], 3))

## 3. Pseudo-likelihood and the missing joint

**Thm 3.** Mask exactly one uniformly chosen position and corrupt nothing else. Then the MLM risk is
$-\tfrac1n\,\mathbb{E}\sum_i \log q(x_i \mid x_{-i})$, which is Besag's pseudo-likelihood. In a family that contains the
truth, the true parameter maximizes the expected log pseudo-likelihood, because each conditional term is maximized
there (Thm 1, term by term). We check the population statement and the estimator's convergence on an Ising chain.

In [ ]:
cm.test_pseudo_likelihood()
i = cm.ising_fits()
print(f"population gradient at (h, J) = (0.2, 0.5): {i['pop_grad']:.2e}")
print(f"PL estimates: N = 200 ({i['h_200']:.6f}, {i['J_200']:.6f}); N = 20000 ({i['h_20000']:.6f}, {i['J_20000']:.6f})")

**Thm 4 (Brook's lemma).** For a positive joint $p$ and any two sequences $x$ and $y$,
$\dfrac{p(x)}{p(y)} = \prod_{i=1}^{n} \dfrac{p(x_i \mid x_{<i}, y_{>i})}{p(y_i \mid x_{<i}, y_{>i})}$. So the full
conditionals determine the joint. We rebuild the HMM's joint over all $25^4$ sequences of length 4 from its conditionals
alone.

**Cor 2 (compatibility).** For two positions $i$ and $j$, the log-ratio $\log p(a, b) - \log p(a', b')$ can be computed
along two paths: $(a,b)\to(a',b)\to(a',b')$, or $(a,b)\to(a,b')\to(a',b')$. Any joint's conditionals give the same answer
both ways, so the gap $\delta$ between the two is $0$. We measure $\delta$ for three sets of conditionals: the HMM's own,
the Bayes-optimal MLM's, and the trained encoder's.

In [ ]:
cm.test_brook()
cm.test_compatibility()
c = cm.compatibility()
for k in ("clean", "bayes", "model"):
    print(f"{k:6} |delta|: median {c[k + '_median']:.3g}, mean {c[k + '_mean']:.3g}, max {c[k + '_max']:.3g}")

The HMM's own conditionals are compatible to rounding. The trained encoder's are not, so no distribution has them
as its conditionals: BERT is not a joint language model. Gibbs-style generation from it and pseudo-log-likelihood
scoring are heuristics built on those conditionals. Even the population optimum is slightly incompatible, because at a
`[MASK]` it predicts the clean token from a context that may itself be corrupted.

## 4. The geometry of contextual vectors

**Def 3 (Ethayarajh).** The *self-similarity* of a word at a layer is the mean cosine between its occurrences' vectors.
The *anisotropy baseline* is the mean cosine between random pairs of vectors of any words. $\mathrm{MEV}$ is
$\sigma_1^2/\sum_i \sigma_i^2$ of the word's occurrence matrix.

**Prop 2.** The mean is the least-squares constant summary of a cloud of occurrence vectors. $\mathrm{MEV}$ lies in
$[1/\mathrm{rank}, 1]$ and equals 1 exactly when the occurrences are collinear (Eckart–Young). It is the largest share of
the cloud's energy that a single static direction can carry.

In [ ]:
cm.test_mev_geometry()
g, fg = cm.hmm_geometry(), cm.filing_geometry()
for l in range(3):
    print(f"layer {l}  HMM w: self-sim {g[f'self_sim_{l}']:.3f}, MEV {g[f'mev_{l}']:.3f}, baseline {g[f'aniso_{l}']:.3f} | "
          f"kestrel: self-sim {fg[f'self_sim_{l}']:.3f}, MEV {fg[f'mev_{l}']:.3f}, baseline {fg[f'aniso_{l}']:.3f}")

## 5. Probing

A linear probe is a ridge-penalized logistic regression on a layer's vectors. Its strength is fixed in advance, never
tuned on a test split. We compare it with two references: Cor 1's static cap, and the exact Bayes accuracy
$\mathbb{E}\max_k P(\text{sense } k \mid x)$ that the HMM makes computable. A control probe is trained and scored on
shuffled senses. A probe that found signal in noise would beat the cap there.

In [ ]:
cm.test_hmm_probe()
for l in range(3):
    print(f"layer {l}: probe {r[f'acc_{l}']:.6f}, shuffled control {r[f'control_{l}']:.6f}")
print(f"static cap {r['static_cap']:.6f}, Bayes {r['bayes']:.6f}, over {r['n_test']} test occurrences")

A probe finds any cue it is given. Read each filing token from document-internal windows, at whichever position lies
nearest the center, and the context-free layer 0 types *kestrel* as company or product through the position table alone:
product mentions close a document, so they land at the end of the last window. Every finance number below reads each token
at the center of its own window instead.

In [ ]:
cm.test_position_leak()
pl = cm.position_leak()
print(f"layer 0 at document-internal positions: 5-fold {pl['cv']:.6f}; PRODUCT positions {pl['product_positions']}")

## 6. Finance case study: two senses, and the news a word reports

We train the same encoder ($d = 64$, two blocks, windows of 32 tokens) on `generate_corpus(300, 0)`, the corpus Topic 7's
SGNS was trained on. Every analyzed token is read at the center of its own window, so layer 0 is exactly static.

**(a) A company or its product.** The corpus reuses most company aliases as product names. *Kestrel* occurs 57 times as
ORG ("Kestrel Turbine Services") and 61 times as PRODUCT ("the Kestrel turbine").

In [ ]:
cm.test_alias_typing()
a = cm.alias_typing()
print(f"21 ambiguous aliases, {a['n']} occurrences ({a['n_product']} PRODUCT); leave-one-alias-out, per-alias majority {a['majority']:.6f}")
print({k: round(a[f"loao_{k}"], 6) for k in ("sgns", "layer0", "layer1", "layer2")})
print("kestrel 5-fold:", {l: round(a[f"kestrel_cv_layer{l}"], 6) for l in range(3)}, "majority", round(a["kestrel_majority"], 6))

**(b) The news a word reports.** In the corpus, "rose" after *revenue* is good news and "rose" after *costs* is bad
news. Likewise "exceeded expectations" is bad news for a cost. Each direction word or evaluative verb that reports a
gold aspect inherits that aspect's polarity. One vector per word can do no better than the word's majority share.

In [ ]:
cm.test_news_polarity()
n = cm.news_polarity()
print(f"{n['n']} labeled occurrences: static cap {n['cap']:.6f}; layer 0 {n['layer0']:.6f}, layer 1 {n['layer1']:.6f}, "
      f"layer 2 {n['layer2']:.6f}; shuffled control {n['control1']:.6f}")
print({w: (v["n"], round(v["layer1"], 3), round(v["cap"], 3)) for w, v in n["words"].items()})

**Topic 7's question.** Can contextual vectors tell *strong* from *weak*? We rerun Topic 7's leave-one-out polarity
probe on each lexicon word's mean vector. Then we ask how much the text carries: the vote of the other lexicon words
near each occurrence.

In [ ]:
cm.test_strong_weak()
s = cm.strong_weak()
print(f"probe (chance 0.5, se {s['chance_se']:.3f}): SGNS {s['sgns']:.6f}, layer 0 {s['layer0']:.6f}, layer 1 {s['layer1']:.6f}, layer 2 {s['layer2']:.6f}")
print(f"neighbors' vote: within 8 {s['vote8']:.6f}, within 32 {s['vote32']:.6f}, whole document {s['vote_doc']:.6f}")
print(f"a lexicon word matches its document's tone {s['word_matches_tone']:.6f} of the time")

A word's polarity is a fact about the dictionary, not about any one sentence, and this corpus barely writes it into
the text: even a whole document's vote gets 0.571. What context does carry is what an occurrence *reports*, and there
the contextual vectors read it. Topic 7 said that one vector per type cannot tell *strong* from *weak* by their windows.
The answer here is more specific: per-occurrence vectors recover what the context determines, and nothing more.

**The encoder at visible words.** With $V = 879$, Theorem 2's optimum keeps nearly all its mass on a visible word.
The filing encoder keeps less: visible positions carry under 3% of its training signal.

In [ ]:
cm.test_filing_visible_shortfall()
ff = cm.filing_fit()
print(f"[MASK] cross-entropy {ff['ce']:.6f} (unigram {ff['unigram_ce']:.6f}); at visible words: encoder {ff['visible_model']:.6f}, "
      f"Theorem 2 {ff['visible_formula']:.6f}, [MASK] guess {ff['visible_masked']:.6f}")

## The lab's data and the harness

In [ ]:
cm.test_viz_payload()
cm.test_viz_mirror_matches_payload()
cm.test_corpus_fingerprint()
print("payload sections:", list(cm.viz_payload()))

In [ ]:
assert cm._run_harness() == 0